# Notebook 01: Ingesta & Bronze Layer

## Objetivos
- Cargar datos crudos (CSV/Parquet) generados sintéticamente
- Validar schema de los datos
- Escribir en Delta Lake (capa Bronze) con partición por fecha
- Benchmark: comparar rendimiento sin optimización vs. con optimización

In [1]:
import sys
sys.path.insert(0, '/home/jovyan')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from delta import configure_spark_with_delta_pip
import time

from src.config import (
    SPARK_APP_NAME, BRONZE_PATH, DATA_RAW,
    PARTITION_COLS_BRONZE, DEFAULT_NUM_RECORDS,
)
from src.utils import benchmark, setup_logging, show_table_info
from src.data_generator import generate_transactions, generate_reference_tables

setup_logging()

## 1. Crear SparkSession

In [2]:
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName(f"{SPARK_APP_NAME}-bronze")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.adaptive.enabled", "true")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Spark UI: http://localhost:4040")

Spark version: 3.5.0
Spark UI: http://localhost:4040


## 2. Generar Datos Sintéticos

Generamos transacciones sintéticas usando PySpark. Para desarrollo/pruebas usar un subset (1M), para el benchmark final usar 100M+.

In [3]:
# Para desarrollo: 100K registros. Para benchmark final: cambiar a DEFAULT_NUM_RECORDS (100M)
NUM_RECORDS = 100_000

with benchmark(f"Generación de {NUM_RECORDS:,} transacciones"):
    transactions_df = generate_transactions(spark, num_records=NUM_RECORDS, num_partitions=4)
    transactions_df.cache()
    count = transactions_df.count()

print(f"\nRegistros generados: {count:,}")

2026-03-26 23:26:03 | src.utils | INFO | ⏱ Iniciando: Generación de 100,000 transacciones
2026-03-26 23:26:09 | src.utils | INFO | ✅ Generación de 100,000 transacciones completado en 6.44 segundos



Registros generados: 100,000


In [4]:
show_table_info(transactions_df, "Transacciones Generadas")
transactions_df.show(5, truncate=False)


📊 Transacciones Generadas
  Filas:    100,000
  Columnas: 16
  Particiones: 4

root
 |-- transaction_id: string (nullable = false)
 |-- account_id: string (nullable = true)
 |-- merchant_id: string (nullable = true)
 |-- transaction_timestamp: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- country: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- is_fraud: integer (nullable = false)
 |-- risk_score: double (nullable = true)

+----------------+------------+------------+---------------------+----------------+----+-----+----+-----------+-----------------+----------------+-------+-------+------+--------+----------+
|transaction_id  |account_id  |merchant_

## 3. Validación de Schema

In [5]:
# Verificar columnas requeridas
required_columns = [
    "transaction_id", "account_id", "merchant_id", "amount",
    "is_fraud", "transaction_timestamp", "transaction_date",
    "year", "month", "merchant_category",
]

missing = set(required_columns) - set(transactions_df.columns)
if missing:
    raise ValueError(f"Columnas faltantes: {missing}")
else:
    print("✅ Todas las columnas requeridas están presentes")

# Verificar nulos en columnas críticas
for col in ["transaction_id", "account_id", "amount"]:
    null_count = transactions_df.where(F.col(col).isNull()).count()
    print(f"  {col}: {null_count} nulos")

# Distribución de fraude
fraud_dist = transactions_df.groupBy("is_fraud").count().orderBy("is_fraud")
fraud_dist.show()

✅ Todas las columnas requeridas están presentes
  transaction_id: 0 nulos
  account_id: 0 nulos
  amount: 0 nulos
+--------+-----+
|is_fraud|count|
+--------+-----+
|       0|98790|
|       1| 1210|
+--------+-----+



## 4. Escritura a Delta Lake (Bronze)

### 4a. Sin Optimización (Benchmark Naïve)

In [6]:
BRONZE_NAIVE_PATH = BRONZE_PATH + "_naive"

with benchmark("Escritura Bronze SIN optimización"):
    (
        transactions_df
        .write
        .format("delta")
        .mode("overwrite")
        .save(BRONZE_NAIVE_PATH)
    )

2026-03-26 23:26:11 | src.utils | INFO | ⏱ Iniciando: Escritura Bronze SIN optimización
2026-03-26 23:26:16 | src.utils | INFO | ✅ Escritura Bronze SIN optimización completado en 5.11 segundos


### 4b. Con Optimización (Particionamiento + ZSTD)

In [7]:
with benchmark("Escritura Bronze CON optimización (partición + zstd)"):
    (
        transactions_df
        .repartition(*[F.col(c) for c in PARTITION_COLS_BRONZE])
        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy(*PARTITION_COLS_BRONZE)
        .option("compression", "zstd")
        .save(BRONZE_PATH)
    )

2026-03-26 23:26:16 | src.utils | INFO | ⏱ Iniciando: Escritura Bronze CON optimización (partición + zstd)
2026-03-26 23:26:18 | src.utils | INFO | ✅ Escritura Bronze CON optimización (partición + zstd) completado en 2.82 segundos


## 5. Verificación de Lectura

In [8]:
bronze_df = spark.read.format("delta").load(BRONZE_PATH)
show_table_info(bronze_df, "Bronze Layer (Delta Lake)")

# Verificar particiones
print("Particiones por año/mes:")
bronze_df.groupBy("year", "month").count().orderBy("year", "month").show()


📊 Bronze Layer (Delta Lake)
  Filas:    100,000
  Columnas: 16
  Particiones: 3

root
 |-- transaction_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- merchant_id: string (nullable = true)
 |-- transaction_timestamp: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- country: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- is_fraud: integer (nullable = true)
 |-- risk_score: double (nullable = true)

Particiones por año/mes:
+----+-----+-----+
|year|month|count|
+----+-----+-----+
|2024|    1|17315|
|2024|    2|15880|
|2024|    3|17182|
|2024|    4|16522|
|2024|    5|17286|
|2024|    6|15815|
+----+-----+-----+



In [9]:
# Generar y guardar tabla de merchants para joins posteriores
merchants_df = generate_reference_tables(spark)
merchants_path = str(DATA_RAW / "merchants")

merchants_df.write.format("parquet").mode("overwrite").save(merchants_path)
print(f"Merchants guardados en: {merchants_path}")
print(f"Total merchants: {merchants_df.count():,}")

Merchants guardados en: /home/jovyan/data/raw/merchants
Total merchants: 50,000


In [10]:
# Liberar cache
transactions_df.unpersist()
print("✅ Notebook 01 completado. Bronze layer lista para transformaciones.")

✅ Notebook 01 completado. Bronze layer lista para transformaciones.
